<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/fully_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [2]:
import os, shutil, tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

In [3]:
BASE = "/content/newdata"
IMG_SRC = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR = "/drive/MyDrive/checkpoints"
MODEL_PATH = "/drive/MyDrive/final_mnv4_edge.keras"

In [4]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)

shutil.copytree(IMG_SRC, BASE)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [5]:
image_size = 224
batch_size = 16

In [6]:
def add_edge_map(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)

    gray = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)

    edge = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))

    # FIX: stable normalization (your previous version caused instability)
    edge = (edge - tf.reduce_min(edge)) / (tf.reduce_max(edge) - tf.reduce_min(edge) + 1e-6)

    return (image, edge), label


def load_ds(path, shuffle):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(image_size, image_size),
        batch_size=batch_size,
        label_mode="categorical",
        shuffle=shuffle
    )
    return ds.map(add_edge_map).prefetch(tf.data.AUTOTUNE)

train_ds = load_ds(f"{BASE}/train", True)
val_ds   = load_ds(f"{BASE}/valid", False)
test_ds  = load_ds(f"{BASE}/test", False)

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [7]:
def create_model():

    rgb_input = layers.Input(shape=(image_size, image_size, 3))
    edge_input = layers.Input(shape=(image_size, image_size, 1))

    # augmentation
    x = layers.RandomFlip("horizontal")(rgb_input)
    x = layers.RandomRotation(0.05)(x)
    x = preprocess_input(x)

    # backbone
    backbone = MobileNetV3Large(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )

    # 🔥 IMPORTANT FIX: partial fine-tuning
    backbone.trainable = True
    for layer in backbone.layers[:-40]:
        layer.trainable = False

    feature_map = backbone.output

    # ======================
    # EDGE BRANCH
    # ======================
    e = layers.Conv2D(32, 3, activation="relu", padding="same")(edge_input)
    e = layers.MaxPooling2D()(e)
    e = layers.Conv2D(64, 3, activation="relu", padding="same")(e)
    e = layers.MaxPooling2D()(e)
    e = layers.Conv2D(128, 3, activation="relu", padding="same")(e)

    e = layers.Resizing(feature_map.shape[1], feature_map.shape[2])(e)
    e = layers.Conv2D(feature_map.shape[-1], 1, padding="same")(e)

    # ======================
    # FUSION (SE ATTENTION)
    # ======================
    fused = layers.Concatenate()([feature_map, e])

    se = layers.GlobalAveragePooling2D()(fused)
    se = layers.Dense(128, activation="relu")(se)
    se = layers.Dense(fused.shape[-1], activation="sigmoid")(se)

    fused = layers.GlobalAveragePooling2D()(fused)
    fused = layers.Multiply()([fused, se])

    # ======================
    # CLASSIFIER
    # ======================
    x = layers.Dense(256, activation="relu")(fused)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(2, activation="softmax")(x)

    model = tf.keras.Model(inputs=[rgb_input, edge_input], outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )

    return model

In [8]:
model = create_model()

12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
callbacks = [
    ModelCheckpoint(
        CHECKPOINT_DIR + "/best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    )
]

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=callbacks
)

Epoch 1/40
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step - accuracy: 0.7508 - loss: 0.5543
Epoch 1: val_accuracy improved from None to 0.88911, saving model to /drive/MyDrive/checkpoints/best.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 162s 282ms/step - accuracy: 0.8436 - loss: 0.4657 - val_accuracy: 0.8891 - val_loss: 0.3988 - learning_rate: 1.0000e-05
Epoch 2/40
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step - accuracy: 0.8945 - loss: 0.3634
Epoch 2: val_accuracy did not improve from 0.88911
501/501 ━━━━━━━━━━━━━━━━━━━━ 121s 241ms/step - accuracy: 0.8887 - loss: 0.3686 - val_accuracy: 0.8891 - val_loss: 0.3750 - learning_rate: 1.0000e-05
Epoch 3/40
500/501 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step - accuracy: 0.8935 - loss: 0.3429
Epoch 3: val_accuracy did not improve from 0.88911
501/501 ━━━━━━━━━━━━━━━━━━━━ 121s 240ms/step - accuracy: 0.8888 - loss: 0.3507 - val_accuracy: 0.8891 - val_loss: 0.3504 - learning_rate: 1.0000e-05
Epoch 4

In [11]:
loss, acc = model.evaluate(test_ds)
print("TEST ACCURACY:", acc)

model.save(MODEL_PATH)

63/63 ━━━━━━━━━━━━━━━━━━━━ 13s 202ms/step - accuracy: 0.8992 - loss: 0.3099
TEST ACCURACY: 0.8992015719413757
